# Notebook 15B — External Analog Coverage, Assay Reliability & Blind-Test Applicability

**Scope:** CPU-first, structure-only applicability audit for the 750-compound OpenADMET CYP inhibition blind test **plus** a carefully nested sensitivity analysis of ChEMBL local correction on the established Notebook 11 out-of-fold (OOF) ensemble.

This notebook reuses the real ChEMBL curation and assay provenance saved in **Notebook 15**. No ChEMBL API redownload or CheMeleon fine-tuning is required.

### Research questions
1. How much better is *blind-set* ChEMBL neighbor coverage than the already-labeled OpenADMET set?
2. Do 2,048-bit Morgan fingerprints change the number and quality of matching CYP-active analogs versus 512 bits?
3. Does excluding high-disagreement assays or requiring replicated measurements improve the local correction?
4. Do stricter similarity thresholds (0.45, 0.55, 0.65, 0.75) help, and do they survive **inner-fold-selected**, held-out outer evaluation?

**Critical distinction:** The 750 test structures have no public potency labels. We can measure similarity and available external reference assays, **not blind-test MA-ST-RAE**. Never use an externally published pIC50 for an *exact blind-test structure* in this notebook's correction models: exact matches are quarantined and separately audited.

Official dataset: https://huggingface.co/datasets/openadmet/cyp-challenge-train-test . Challenge: https://openadmet.org/blindchallenges/ .


## 0. Install (Colab CPU)

In [ ]:
%pip -q install "rdkit>=2023.9" "pandas>=2.0" "numpy>=1.26" "requests>=2.31" "ipython>=8"

## 1. Configure inputs and resumable cache

Upload **`cyp_notebook_11_results.zip`** and **`cyp_notebook_15_results.zip`** to Colab. You can optionally upload the *official* `cyp-challenge-TEST-BLINDED.csv`; otherwise it is downloaded from the OpenADMET Hugging Face dataset. The notebook will reject labels and the wrong structure-track test file.

Run the small `FAST_MODE=True` check first. Set `FAST_MODE=False` for **all five folds and all 750 blind structures**. A persistent Google Drive path is recommended for the more expensive 2,048-bit similarity scans. Never mix cache folders between experiments without their ZIP signatures.


In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np, pandas as pd
FAST_MODE = True
NB11_RESULTS_ZIP = '/content/cyp_notebook_11_results.zip'
NB15_RESULTS_ZIP = '/content/cyp_notebook_15_results.zip'
BLIND_CSV = None  # None -> download the official 750-row blinded inhibition CSV
AUTO_DOWNLOAD_BLIND = True
REQUIRE_750_BLIND = True
PERSISTENT_ROOT = '/content/cyp_notebook_15B'
# For Drive persistence:
# from google.colab import drive; drive.mount('/content/drive')
# PERSISTENT_ROOT = '/content/drive/MyDrive/CYP/Notebook15B'
FP_BITS = (512,2048)
TOP_K_STRUCTURES = 5
SEED = 2026
FOLDS_TO_RUN = [0] if FAST_MODE else [0,1,2,3,4]
MAX_FAST_ROWS_PER_ISO = 22
MAX_FAST_BLIND = 30
MIN_INNER_GAIN = 0.0005 # regularize selection: stay with baseline for negligible gains
N_BOOTSTRAP = 500  # full-run only
BASE = Path(PERSISTENT_ROOT)
CACHE = BASE / 'cache'
OUT = BASE / 'results'
CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
# Convenient path resolution for local notebooks/test environments
for key in ('NB11_RESULTS_ZIP','NB15_RESULTS_ZIP'):
    p=Path(globals()[key])
    if not p.exists():
        for alt in (Path.cwd()/p.name,Path('/mnt/data')/p.name):
            if alt.exists():globals()[key]=str(alt);break
if not Path(NB11_RESULTS_ZIP).exists():raise FileNotFoundError('Upload real Notebook 11 results ZIP')
if not Path(NB15_RESULTS_ZIP).exists():raise FileNotFoundError('Upload completed Notebook 15 results ZIP')
print('Mode:', 'FAST / incomplete' if FAST_MODE else 'FULL / five-fold')


## 2. Self-contained chemistry, scoring, and leakage-safe functions

The implementation below contains the canonicalization, fingerprint searches, cross-assay reliability filters, fold-specific assay offsets, nested selection, and scaffold-block bootstrap. **All ChEMBL IC50 values remain identified by assay.** `assay_mad==0` for a single record is **not** treated as proof that the assay is intrinsically reliable.


In [ ]:
"""Notebook 15B self-contained ChEMBL analog coverage and nested reliability audit.
Uses only structure data for the blind test, and only inner training labels for selection.
"""
import hashlib, io, json, os, zipfile, math
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog('rdApp.warning')

ISOFORMS=('CYP1A2','CYP2C9','CYP2D6','CYP3A4')
DEFAULT_BLIND_URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/cyp-challenge-TEST-BLINDED.csv'
QUALITY_TIERS=('all','concordant','replicated')
THRESHOLDS=(0.45,0.55,0.65,0.75)
STRENGTHS=(0.15,0.30,0.50)

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()

def read_member(z, filename):
    names=[x for x in z.namelist() if x == filename or x.endswith('/'+filename)]
    if len(names)!=1:raise ValueError('Expected one archive member '+filename+'; got '+repr(names))
    return z.read(names[0])

def canonical(s):
    if not isinstance(s,str) or not s.strip():return None
    m=Chem.MolFromSmiles(s)
    if m is None:return None
    return Chem.MolToSmiles(m,isomericSmiles=True,canonical=True)

def load_inputs(nb11_zip,nb15_zip):
    with zipfile.ZipFile(nb11_zip) as z:
        meta11=json.loads(read_member(z,'run_metadata.json'))
        if not(meta11.get('complete_five_fold_oof') and meta11.get('scientifically_valid_full_model_run')):
            raise RuntimeError('Notebook 11 results must be a full five-fold real CheMeleon run')
        outer=pd.read_csv(io.BytesIO(read_member(z,'nested_outer_predictions.csv')))
        inner=pd.read_csv(io.BytesIO(read_member(z,'inner_oof_predictions_for_selection.csv')))
        manifest=pd.read_csv(io.BytesIO(read_member(z,'fold_manifest.csv')))
    with zipfile.ZipFile(nb15_zip) as z:
        meta15=json.loads(read_member(z,'run_metadata.json'))
        if not meta15.get('full_scientific_evaluation'):
            raise RuntimeError('Notebook 15 archive must be a completed non-synthetic run')
        external=pd.read_csv(io.BytesIO(read_member(z,'chembl_external_references.csv')))
        overlap=pd.read_csv(io.BytesIO(read_member(z,'chembl_excluded_exact_overlaps.csv')))
        disagreement=pd.read_csv(io.BytesIO(read_member(z,'chembl_cross_assay_disagreement.csv')))
    assert meta15['nb11_sha256']==sha_file(nb11_zip),'Notebook 15 and 11 zip SHA mismatch'
    model='CheMeleon_finetuned + ScreenStack_LGBM'
    outer=outer.loc[outer.subset==model].rename(columns={'y_pred':'baseline'}).copy()
    assert len(outer)==6525 and len(manifest)==4905
    assert not outer.duplicated(['isoform','canonical_smiles']).any()
    assert set(outer.isoform)==set(ISOFORMS)
    inner['baseline']=0.5*(inner.CheMeleon_finetuned+inner.ScreenStack_LGBM)
    bounds=outer[['canonical_smiles','isoform','ci_low','ci_high']].drop_duplicates()
    inner=inner.merge(bounds,on=['canonical_smiles','isoform'],validate='many_to_one')
    inner=inner.merge(manifest[['canonical_smiles','scaffold_group']],on='canonical_smiles',validate='many_to_one')
    assert inner[['baseline','ci_low','ci_high']].notna().all().all()
    assert not inner.duplicated(['fold','isoform','canonical_smiles']).any()
    assert not manifest.duplicated('canonical_smiles').any()
    external['pIC50']=pd.to_numeric(external.pIC50,errors='coerce')
    overlap['pIC50']=pd.to_numeric(overlap.pIC50,errors='coerce')
    for name,frame in [('external',external),('overlap',overlap)]:
        if 'cross_assay_span' not in frame:
            frame=frame.merge(disagreement[['isoform','canonical_smiles','cross_assay_span']],
                              on=['isoform','canonical_smiles'],how='left',validate='many_to_one')
        if name=='external':external=frame
        else:overlap=frame
    return meta11,meta15,outer,inner,manifest,external,overlap,disagreement

def validate_blind(df, require_750=True):
    if not {'Molecule_Name','SMILES'} <= set(df):
        raise ValueError('Blind file must have Molecule_Name and SMILES')
    other=[c for c in df.columns if c not in ['Molecule_Name','SMILES']]
    if any(x in c.lower() for c in other for x in ['ic50','pic50','activity','inhib','confidence','ci_','label']):
        raise ValueError('BLIND STRUCTURES ONLY: file contains potentially labeled columns')
    b=df[['Molecule_Name','SMILES']].copy()
    b['canonical_smiles']=b.SMILES.map(canonical)
    if b.canonical_smiles.isna().any():raise ValueError('Blind input has unparseable structures')
    if b.Molecule_Name.isna().any() or b.Molecule_Name.duplicated().any():
        raise ValueError('Missing/duplicated blind identifiers')
    if require_750 and len(b)!=750:raise ValueError('Expected exactly 750 blinded inhibition structures, got '+str(len(b)))
    return b

def load_blind(path,cache,download=True,require_750=True):
    p=Path(path) if path else Path(cache)/'cyp-challenge-TEST-BLINDED.csv'
    if not p.exists():
        if not download:raise FileNotFoundError(str(p))
        import requests
        r=requests.get(DEFAULT_BLIND_URL,timeout=(20,120));r.raise_for_status()
        if not r.content.startswith(b'Molecule_Name') and b'SMILES' not in r.content[:1024]:
            raise ValueError('Blind download is not a CSV with the expected schema')
        p.parent.mkdir(parents=True,exist_ok=True)
        p.write_bytes(r.content)
    return validate_blind(pd.read_csv(p),require_750=require_750),str(p)

def quality_mask(frame,tier):
    rec=pd.to_numeric(frame.n_records,errors='coerce').fillna(0)
    mad=pd.to_numeric(frame.assay_mad,errors='coerce').fillna(np.inf)
    # Missing cross-assay spans represent a single observed assay, NOT evidence of high reliability.
    span=pd.to_numeric(frame.cross_assay_span,errors='coerce').fillna(0)
    if tier=='all':return (rec>=1) & np.isfinite(frame.pIC50)
    if tier=='concordant':return (rec>=1) & (mad<=0.30) & (span<=0.50) & np.isfinite(frame.pIC50)
    if tier=='replicated':return (rec>=2) & (mad<=0.30) & (span<=0.50) & np.isfinite(frame.pIC50)
    raise KeyError(tier)

def prepare_refs(external,blind_smiles):
    raw=external.copy()
    raw['canonical_smiles']=raw.canonical_smiles.astype(str)
    bad=raw.canonical_smiles.isin(set(blind_smiles))
    exact=raw[bad].copy()
    # Conservative: never use external labels on exact blind structures as training/reference activities.
    eligible=raw[~bad].copy()
    return eligible,exact

def fingerprint_cache(smis,bits):
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=int(bits))
    out={}
    for s in sorted(set(smis)):
        m=Chem.MolFromSmiles(s)
        if m is not None:out[s]=gen.GetFingerprint(m)
    return out

def build_neighbors(iso,bits,queries,external,k=5):
    """Exact top-k distinct-structure neighbor search; preindexed assay records.
    Runtime scales with fingerprint comparisons, not pandas row indexing in the query loop.
    """
    ref=external[external.isoform.eq(iso)].reset_index(drop=True).copy()
    cols=['isoform','bits','tier','query_smiles','reference_smiles','assay_chembl_id','pIC50','sim','rank','quality_score']
    if ref.empty:return pd.DataFrame(columns=cols)
    rec=pd.to_numeric(ref.n_records,errors='coerce').fillna(0).to_numpy(float)
    mad=pd.to_numeric(ref.assay_mad,errors='coerce').fillna(np.inf).to_numpy(float)
    span=pd.to_numeric(ref.cross_assay_span,errors='coerce').fillna(0).to_numpy(float)
    quality=1.0/(1+np.clip(mad,0,3)+.5*np.clip(span,0,5))
    yes={tier:quality_mask(ref,tier).to_numpy(bool) for tier in QUALITY_TIERS}
    structs=ref.canonical_smiles.astype(str).to_numpy()
    assays=ref.assay_chembl_id.astype(str).to_numpy()
    activities=ref.pIC50.to_numpy(float)
    unique=list(dict.fromkeys(structs))
    fp=fingerprint_cache(set(queries)|set(unique),bits)
    unique=[one for one in unique if one in fp]
    fp_arr=[fp[x] for x in unique]
    index={s:i for i,s in enumerate(unique)}
    by_smiles=defaultdict(list)
    for j,s in enumerate(structs):
        if s in index:by_smiles[s].append(j)
    valid_by_tier={tier:np.asarray([i for i,s in enumerate(unique)
                       if np.any(yes[tier][by_smiles[s]])],dtype=np.int32)
                   for tier in QUALITY_TIERS}
    out=[]
    for s in sorted(set(queries)):
        if s not in fp:continue
        sims=np.asarray(DataStructs.BulkTanimotoSimilarity(fp[s],fp_arr),dtype=np.float32)
        for tier in QUALITY_TIERS:
            valid=valid_by_tier[tier]
            if len(valid)==0:continue
            sim_valid=sims[valid]
            if len(valid)>k:
                top=valid[np.argpartition(-sim_valid,k-1)[:k]]
                top=top[np.argsort(-sims[top],kind='stable')]
            else:
                top=valid[np.argsort(-sim_valid,kind='stable')]
            for rank,i in enumerate(top,1):
                reference=unique[int(i)]
                sim=float(sims[i])
                for j in by_smiles[reference]:
                    if not yes[tier][j]:continue
                    out.append((iso,bits,tier,s,reference,assays[j],activities[j],sim,rank,float(quality[j])))
    return pd.DataFrame(out,columns=cols)

def neighbor_lookup(df):
    lookup={}
    for key,g in df.groupby(['isoform','bits','tier','query_smiles'],sort=False):
        lookup[key]=[(float(r.sim),str(r.reference_smiles),str(r.assay_chembl_id),float(r.pIC50),float(r.quality_score))
                     for r in g.itertuples(index=False)]
    return lookup

def calibrate_offsets(overlap, training, iso, min_assay=2,min_global=3):
    # Primary labels come ONLY from the current inner/outer fitting partition.
    tr=training[training.isoform.eq(iso)][['canonical_smiles','y_true']].drop_duplicates()
    ov=overlap[overlap.isoform.eq(iso)]
    j=ov.merge(tr,on='canonical_smiles',how='inner',validate='many_to_one')
    if j.empty:return {},np.nan,0
    j['offset']=j.y_true-j.pIC50
    assays={a:float(v.groupby('canonical_smiles').offset.median().median())
            for a,v in j.groupby('assay_chembl_id') if v.canonical_smiles.nunique()>=min_assay}
    glob=float(j.groupby('canonical_smiles').offset.median().median()) if j.canonical_smiles.nunique()>=min_global else np.nan
    return assays,glob,j.canonical_smiles.nunique()

def corrected_row(base,neighbors,offsets,threshold,strength):
    by_assay,global_offset,_=offsets
    # Average assay offsets within each neighbor structure before averaging structures.
    per_mol=defaultdict(lambda:[[],[]])
    maxsim=max((r[0] for r in neighbors),default=0.)
    for sim,smiles,assay,y,q in neighbors:
        if sim<threshold:continue
        off=by_assay.get(assay,global_offset)
        if not np.isfinite(off):continue
        per_mol[smiles][0].append(float(y+off))
        per_mol[smiles][1].append(float(q))
    points=[]; weights=[]
    for smi,(ys,qs) in per_mol.items():
        w=np.asarray(qs,float)
        points.append(float(np.average(ys,weights=w)))
        sim=max(r[0] for r in neighbors if r[1]==smi)
        weights.append(float(sim**5 * np.mean(qs)))
    if not points:return float(base),0,maxsim
    local=float(np.average(points,weights=weights))
    gate=float(np.clip((maxsim-threshold)/0.30,0,1)*min(1.0,len(points)/3))
    delta=float(np.clip(local-float(base),-1.5,1.5))
    return float(base+strength*gate*delta),len(points),maxsim

def metric(frame,model,group='isoform'):
    vals=[]
    for iso,g in frame.groupby(group):
        y=g.y_true.to_numpy(float);pred=g[model].to_numpy(float)
        lo=g.ci_low.to_numpy(float);hi=g.ci_high.to_numpy(float)
        num=np.maximum(lo-pred,0)+np.maximum(pred-hi,0)
        midpoint=y.mean()
        den=np.maximum(lo-midpoint,0)+np.maximum(midpoint-hi,0)
        score=float(num.sum()/den.sum()) if den.sum()>1e-10 else np.nan
        vals.append({'isoform':iso,'n':len(g),'mae':float(np.mean(np.abs(y-pred))), 'st_rae':score})
    return pd.DataFrame(vals)

def macro_metrics(frame,model):
    per=metric(frame,model)
    if len(per)!=4 or not np.isfinite(per.st_rae).all():raise ValueError('Missing isoform/invalid denominator')
    return float(per.st_rae.mean()),float(per.mae.mean())

def candidate_name(bits,tier,threshold,strength):
    return f'b{bits}_{tier}_t{threshold:.2f}_w{strength:.2f}'

def predictions_on_split(val,training,overlap,lookups,fp_bits,tiers,thresholds,strengths,
                         audit,outer_fold,stage,inner_fold=None):
    if not set(val.canonical_smiles).isdisjoint(set(training.canonical_smiles)):
        raise AssertionError('Validation molecules in training anchors')
    if not set(val.scaffold_group).isdisjoint(set(training.scaffold_group)):
        raise AssertionError('Validation scaffolds in calibration training partition')
    output=val[['canonical_smiles','isoform','scaffold_group','y_true','baseline','ci_low','ci_high']].copy().reset_index(drop=True)
    offsets={(tier,iso):calibrate_offsets(overlap[quality_mask(overlap,tier)],training,iso)
             for tier in tiers for iso in ISOFORMS}
    for iso in ISOFORMS:
        qset=set(val[val.isoform.eq(iso)].canonical_smiles)
        tset=set(training[training.isoform.eq(iso)].canonical_smiles)
        for tier in tiers:
            audit.append({'stage':stage,'fold':outer_fold,'inner_fold':inner_fold,'isoform':iso,
                'tier':tier,'n_query':len(qset),'n_fit':len(tset),
                'molecule_overlap':len(qset&tset),
                'scaffold_overlap':len(set(val.scaffold_group)&set(training.scaffold_group)),
                'calibration_overlap_structures':int(offsets[(tier,iso)][2])})
    groups=output.groupby('isoform',sort=False).groups
    extra={}
    for bits in fp_bits:
        for tier in tiers:
            for threshold in thresholds:
                deltas=np.zeros(len(output),dtype=float)
                n_refs=np.zeros(len(output),dtype=int)
                sims=np.zeros(len(output),dtype=float)
                for iso,ixs in groups.items():
                    off=offsets[(tier,iso)]
                    for i in ixs:
                        r=output.iloc[i]
                        ne=lookups.get((iso,bits,tier,r.canonical_smiles),[])
                        p,n,s=corrected_row(r.baseline,ne,off,threshold,1.0)
                        deltas[i]=p-r.baseline
                        n_refs[i]=n
                        sims[i]=s
                for w in strengths:
                    name=candidate_name(bits,tier,threshold,w)
                    extra[name]=(output.baseline.to_numpy(float)+w*deltas)
                extra[f'support_b{bits}_{tier}_t{threshold:.2f}']=n_refs
                extra[f'nearest_b{bits}_{tier}']=sims
    return pd.concat([output,pd.DataFrame(extra,index=output.index)],axis=1)

def get_summary(frame,candidates):
    scores=[]
    for c in candidates:
        st,mae=macro_metrics(frame,c)
        scores.append({'candidate':c,'macro_st_rae':st,'macro_mae':mae,
                       'fraction_changed':float(np.mean(np.abs(frame[c]-frame.baseline)>1e-8))})
    return pd.DataFrame(scores).sort_values(['macro_st_rae','macro_mae','candidate'],kind='stable').reset_index(drop=True)

def bootstrap_scaffold(frame,reps=500,seed=2026):
    rng=np.random.default_rng(seed)
    groups,inv=np.unique(frame.scaffold_group.astype(str),return_inverse=True)
    y=frame.y_true.to_numpy(float);lo=frame.ci_low.to_numpy(float);hi=frame.ci_high.to_numpy(float)
    pred0=frame.baseline.to_numpy(float);pred1=frame.selected.to_numpy(float)
    iso=frame.isoform.to_numpy(str)
    records=[]
    for rep in range(reps):
        weights=rng.multinomial(len(groups),np.ones(len(groups))/len(groups))[inv].astype(float)
        gains=[];maes=[]
        for one in ISOFORMS:
            m=(iso==one)&(weights>0)
            w=weights[m]
            if not len(w):break
            ym=y[m];center=np.average(ym,weights=w)
            denom=np.dot(w,np.maximum(lo[m]-center,0)+np.maximum(center-hi[m],0))
            if denom<=1e-10:break
            a=np.maximum(lo[m]-pred0[m],0)+np.maximum(pred0[m]-hi[m],0)
            b=np.maximum(lo[m]-pred1[m],0)+np.maximum(pred1[m]-hi[m],0)
            gains.append(float(np.dot(w,a-b)/denom))
            maes.append(float(np.average(np.abs(ym-pred0[m])-np.abs(ym-pred1[m]),weights=w)))
        if len(gains)==4:records.append({'bootstrap':rep,'st_rae_gain':np.mean(gains),'mae_gain':np.mean(maes)})
    return pd.DataFrame(records)

def summary_coverage(neighbors,query_sets,exact_lookup,tiers,fp_bits,thresholds):
    by={}
    # by iso/bits/tier/query, choose highest of all assay rows.
    if len(neighbors):
        by=neighbors.groupby(['isoform','bits','tier','query_smiles'],sort=False).sim.max().to_dict()
    rows=[]; perquery=[]
    for dataset,queries in query_sets.items():
        for iso,query_list in queries.items():
            qs=sorted(set(query_list));n=len(qs)
            if not n:continue
            for bits in fp_bits:
                for tier in tiers:
                    sims=np.asarray([by.get((iso,bits,tier,s),0) for s in qs],float)
                    exact=[s in exact_lookup.get(iso,set()) for s in qs]
                    row={'dataset':dataset,'isoform':iso,'bits':bits,'tier':tier,'n':n,
                         'n_exact_chembl':int(sum(exact)),
                         'median_nearest_similarity':float(np.median(sims)),
                         'mean_nearest_similarity':float(np.mean(sims))}
                    for t in thresholds:
                        row[f'n_ge_{t:.2f}']=int(np.count_nonzero(sims>=t))
                        row[f'fraction_ge_{t:.2f}']=float(np.mean(sims>=t))
                    rows.append(row)
                    for s,sim,ex in zip(qs,sims,exact):
                        perquery.append({'dataset':dataset,'isoform':iso,'canonical_smiles':s,
                             'bits':bits,'tier':tier,'nearest_similarity':sim,'exact_chmbl':ex})
    return pd.DataFrame(rows),pd.DataFrame(perquery)


## 3. Load records, quarantine exact blind matches, and compute analog coverage

Three provenance filters are compared: **all exact IC50**, **concordant** (within-assay median absolute deviation ≤0.30 and measured cross-assay span ≤0.50 pIC50), and **replicated concordant** (same criteria plus at least two records in that compound-assay aggregate). The concordant filter does not prove single-assay records are reproducible: it only removes observed disagreements. In the stricter sensitivity analysis, missing assay replicates are not silently imputed.

The fingerprint scan uses **Morgan radius 2** with either 512 or 2,048 bits. These are different encodings; their numerical Tanimoto thresholds are *not identical molecular distance scales*. Interpret across-fingerprint comparisons descriptively.


In [ ]:
meta11,meta15,OUTER,INNER,MANIFEST,EXTERNAL_RAW,OVERLAP,DISAGREEMENT=load_inputs(NB11_RESULTS_ZIP,NB15_RESULTS_ZIP)
BLIND,BLIND_SOURCE=load_blind(BLIND_CSV,CACHE,download=AUTO_DOWNLOAD_BLIND,require_750=REQUIRE_750_BLIND)
assert set(OUTER.isoform)==set(ISOFORMS)
ALL_PRIMARY=set(MANIFEST.canonical_smiles)
EXTERNAL,BLIND_EXACT=prepare_refs(EXTERNAL_RAW,BLIND.canonical_smiles)
assert not set(EXTERNAL.canonical_smiles)&ALL_PRIMARY
assert not set(EXTERNAL.canonical_smiles)&set(BLIND.canonical_smiles)
assert not set(OVERLAP.canonical_smiles)-ALL_PRIMARY
print('OpenADMET labeled structures:',len(MANIFEST),'| observed pIC50:',len(OUTER))
print('Official blind structures:',len(BLIND),'| exact matches in ChEMBL references:',BLIND_EXACT.canonical_smiles.nunique())
print('External assay rows eligible for corrections:',len(EXTERNAL))

SIGNATURE={'version':'15B_v5','nb11':sha_file(NB11_RESULTS_ZIP),'nb15':sha_file(NB15_RESULTS_ZIP),
           'blind':sha_file(BLIND_SOURCE),'fast_mode':FAST_MODE,'fp_bits':FP_BITS,
           'quality_tiers':QUALITY_TIERS,'thresholds':THRESHOLDS,'strengths':STRENGTHS,
           'seed':SEED,'folds':FOLDS_TO_RUN,'fast_rows':MAX_FAST_ROWS_PER_ISO,
           'quarantine_exact_blind':True}
RUN_HASH=hashlib.sha256(json.dumps(SIGNATURE,sort_keys=True).encode()).hexdigest()[:20]
RC=CACHE/('run_'+RUN_HASH);RC.mkdir(parents=True,exist_ok=True)
print('Cache signature:',RUN_HASH)

# Report quality coverage independently from predictive performance.
QA=[]
for iso in ISOFORMS:
    r=EXTERNAL_RAW[EXTERNAL_RAW.isoform.eq(iso)]
    for tier in QUALITY_TIERS:
        subset=r[quality_mask(r,tier)]
        QA.append({'isoform':iso,'tier':tier,'records':len(subset),
                   'unique_structures':subset.canonical_smiles.nunique(),
                   'n_assays':subset.assay_chembl_id.nunique(),
                   'n_with_replicated_measurements':int((pd.to_numeric(subset.n_records,errors='coerce')>=2).sum()),
                   'n_exact_blind_quarantined':int(subset.canonical_smiles.isin(set(BLIND.canonical_smiles)).sum())})
QUALITY_AUDIT=pd.DataFrame(QA)
QUALITY_AUDIT.to_csv(OUT/'assay_quality_filter_audit.csv',index=False)
BLIND_EXACT.to_csv(OUT/'blind_exact_chembl_quarantine.csv',index=False)

# Query only the publicly provided structures (no blind potency data).
QUERY_BY_ISO={}
for iso in ISOFORMS:
    if FAST_MODE:
        q=set()
        for f in FOLDS_TO_RUN:
            outer_group=OUTER[(OUTER.fold.eq(f)) & (OUTER.isoform.eq(iso))]
            q.update(outer_group.sample(n=min(len(outer_group),MAX_FAST_ROWS_PER_ISO),
                 random_state=SEED+int(f)).canonical_smiles)
            for h in sorted(INNER[INNER.fold.eq(f)].inner_fold.unique()):
                inner_group=INNER[(INNER.fold.eq(f)) & (INNER.inner_fold.eq(h)) &
                                  (INNER.isoform.eq(iso))]
                q.update(inner_group.sample(n=min(len(inner_group),MAX_FAST_ROWS_PER_ISO),
                       random_state=SEED+int(h)).canonical_smiles)
        blind_q=BLIND.canonical_smiles.tolist()[:MAX_FAST_BLIND]
    else:
        q=set(OUTER[OUTER.isoform.eq(iso)].canonical_smiles)
        blind_q=BLIND.canonical_smiles.tolist()
    QUERY_BY_ISO[iso]=q|set(blind_q)


In [ ]:
NEIGH=[]
for iso in ISOFORMS:
    for bits in FP_BITS:
        p=RC/f'neighbors_{iso}_b{bits}.csv'
        if p.exists():
            frame=pd.read_csv(p)
            print('Cached neighbor search:',iso,bits,len(frame))
        else:
            frame=build_neighbors(iso,bits,QUERY_BY_ISO[iso],EXTERNAL,k=TOP_K_STRUCTURES)
            frame.to_csv(p,index=False,float_format='%.16g')
            frame=pd.read_csv(p)
            print('Calculated neighbor search:',iso,bits,len(frame))
        NEIGH.append(frame)
NEIGH=pd.concat(NEIGH,ignore_index=True) if NEIGH else pd.DataFrame()
NEIGH.to_csv(OUT/'top_chembl_neighbors_by_query.csv',index=False,float_format='%.10g')
LOOKUP=neighbor_lookup(NEIGH) if len(NEIGH) else {}

EXACT_BY_ISO={iso:set(EXTERNAL_RAW.loc[EXTERNAL_RAW.isoform.eq(iso),'canonical_smiles']) for iso in ISOFORMS}
SETS={
    'labeled_openadmet':{iso:set(OUTER.loc[OUTER.isoform.eq(iso),'canonical_smiles']) for iso in ISOFORMS},
    'blind_test':{iso:set(BLIND.canonical_smiles[:MAX_FAST_BLIND] if FAST_MODE else BLIND.canonical_smiles) for iso in ISOFORMS},
}
COVERAGE_SUMMARY,COVERAGE_DETAILS=summary_coverage(NEIGH,SETS,EXACT_BY_ISO,QUALITY_TIERS,FP_BITS,THRESHOLDS)
COVERAGE_SUMMARY.to_csv(OUT/'coverage_comparison_train_vs_blind.csv',index=False)
COVERAGE_DETAILS.to_csv(OUT/'coverage_by_compound_isoform.csv',index=False)
BLIND_COV=COVERAGE_DETAILS[COVERAGE_DETAILS.dataset.eq('blind_test')].merge(
    BLIND[['Molecule_Name','canonical_smiles']],on='canonical_smiles',how='left',validate='many_to_many')
BLIND_COV.to_csv(OUT/'blind_chembl_coverage_by_isoform.csv',index=False)
BLIND_BEST=(BLIND_COV.groupby(['Molecule_Name','canonical_smiles'],as_index=False)
            .agg(best_eligible_similarity=('nearest_similarity','max'),
                 exact_chmbl_any_isoform=('exact_chmbl','max')))
BLIND_BEST.to_csv(OUT/'blind_chembl_coverage_by_compound.csv',index=False)
print('Blind >0.75 neighbor coverage (2048-bit, all assays):')
display(COVERAGE_SUMMARY[(COVERAGE_SUMMARY.dataset=='blind_test') & (COVERAGE_SUMMARY.bits==2048) &
   (COVERAGE_SUMMARY.tier=='all')][['isoform','n','median_nearest_similarity','fraction_ge_0.75','n_exact_chembl']])



## 4. Fully nested correction selection on actual Notebook 11 predictions

For each outer fold, assay-offset calibration uses only the training partition. Candidate fingerprint, reliability tier, similarity threshold and correction strength are scored using **inner OOF predictions**; the chosen candidate is evaluated only after selection on that outer fold. A minimum inner-ST-RAE gain of 0.0005 is required to select any correction. The original ensemble is a valid fallback.

**No graph model is retrained.** The baseline is the actual Notebook 11 CheMeleon + screen-stacked OOF predictions. The outer-fold sensitivity table of *unselected* candidates is labeled exploratory because comparing them on the outer targets is not nested model selection.


In [ ]:
# The full nested selection produces valid, untouched outer fold estimates.
ALL_CANDIDATES=['baseline']+[candidate_name(b,t,th,w)
  for b in FP_BITS for t in QUALITY_TIERS for th in THRESHOLDS for w in STRENGTHS]
SCORES=[];SELECTIONS=[];PREDICTIONS=[];LEAKAGE=[];OUTER_SENS=[]
for f in FOLDS_TO_RUN:
    train_all=INNER[INNER.fold.eq(f)].copy()
    outer_all=OUTER[OUTER.fold.eq(f)].copy()
    cache_pred=RC/f'outer_predictions_fold{f}.csv'
    cache_sel=RC/f'selection_fold{f}.json'
    cache_scores=RC/f'inner_scores_fold{f}.csv'
    cache_audit=RC/f'audit_fold{f}.json'
    if all(p.exists() for p in [cache_pred,cache_sel,cache_scores,cache_audit]):
        op=pd.read_csv(cache_pred)
        selection=json.loads(cache_sel.read_text())
        sc=pd.read_csv(cache_scores)
        LEAKAGE.extend(json.loads(cache_audit.read_text()))
        print('Resumed nested fold',f)
    else:
        audit_start=len(LEAKAGE)
        inner_frames=[]
        for h in sorted(train_all.inner_fold.unique()):
            val=train_all[train_all.inner_fold.eq(h)].copy()
            train=train_all[train_all.inner_fold.ne(h)].copy()
            if FAST_MODE:
                val=(val.groupby('isoform',group_keys=False).sample(
                    n=min(MAX_FAST_ROWS_PER_ISO,int(val.groupby('isoform').size().min())),
                    random_state=SEED+int(h)).reset_index(drop=True))
            pi=predictions_on_split(val,train,OVERLAP,LOOKUP,FP_BITS,QUALITY_TIERS,
                 THRESHOLDS,STRENGTHS,LEAKAGE,f,'inner',int(h))
            inner_frames.append(pi)
        ip=pd.concat(inner_frames,ignore_index=True)
        sc=get_summary(ip,ALL_CANDIDATES)
        winner=sc.iloc[0]
        baseline=sc[sc.candidate.eq('baseline')].iloc[0]
        selected=str(winner.candidate) if (baseline.macro_st_rae-winner.macro_st_rae)>=MIN_INNER_GAIN else 'baseline'
        selection={'fold':f,'selected_candidate':selected,
             'inner_best_candidate':str(winner.candidate),
             'inner_selected_st_rae':float(sc[sc.candidate.eq(selected)].macro_st_rae.iloc[0]),
             'inner_baseline_st_rae':float(baseline.macro_st_rae),
             'inner_best_gain':float(baseline.macro_st_rae-winner.macro_st_rae)}
        val=outer_all.copy()
        if FAST_MODE:
            val=val.groupby('isoform',group_keys=False).sample(
                n=min(MAX_FAST_ROWS_PER_ISO,int(val.groupby('isoform').size().min())),
                random_state=SEED+int(f)).reset_index(drop=True)
        op=predictions_on_split(val,train_all,OVERLAP,LOOKUP,FP_BITS,QUALITY_TIERS,
                     THRESHOLDS,STRENGTHS,LEAKAGE,f,'outer',None)
        op['selected']=op[selected]
        op['fold']=f
        # Cache must include the original selection and both audit stages.
        sc.to_csv(cache_scores,index=False,float_format='%.16g')
        sc=pd.read_csv(cache_scores)
        op.to_csv(cache_pred,index=False,float_format='%.16g')
        cache_sel.write_text(json.dumps(selection,indent=2))
        cache_audit.write_text(json.dumps(LEAKAGE[audit_start:],indent=2,default=str))
        op=pd.read_csv(cache_pred)
    if 'selected' not in op:raise AssertionError('Selected prediction missing in cached outer fold')
    if len(op) != len(op[['canonical_smiles','isoform']].drop_duplicates()):raise AssertionError('Duplicated outer rows')
    SCORES.append(sc.assign(fold=f))
    SELECTIONS.append(selection)
    PREDICTIONS.append(op)
    per_fold=get_summary(op,['baseline','selected'])
    OUTER_SENS.append(per_fold.assign(fold=f))
    print('Fold',f,'chosen',selection['selected_candidate'],
          '| ST-RAE',round(float(per_fold[per_fold.candidate.eq('selected')].macro_st_rae.iloc[0]),5))

INNER_SCORES=pd.concat(SCORES,ignore_index=True)
SELECTED_PARAMS=pd.DataFrame(SELECTIONS)
OUTER_ALL=pd.concat(PREDICTIONS,ignore_index=True)
LEAK_AUDIT=pd.DataFrame(LEAKAGE)
assert (LEAK_AUDIT.molecule_overlap==0).all() and (LEAK_AUDIT.scaffold_overlap==0).all()
INNER_SCORES.to_csv(OUT/'nested_inner_candidate_scores.csv',index=False)
SELECTED_PARAMS.to_csv(OUT/'nested_selected_params_by_fold.csv',index=False)
LEAK_AUDIT.to_csv(OUT/'leakage_audit.csv',index=False)
selected_cols=['fold','canonical_smiles','isoform','scaffold_group','y_true','baseline','selected','ci_low','ci_high']
OUTER_ALL[selected_cols].to_csv(OUT/'nested_outer_15B_predictions.csv',index=False,float_format='%.16g')
if not FAST_MODE:
    assert len(OUTER_ALL)==6525,('Missing OOF predictions',len(OUTER_ALL))

MODEL_SUMMARY=pd.DataFrame([{'model':name,'macro_st_rae':macro_metrics(OUTER_ALL,name)[0],
                             'macro_mae':macro_metrics(OUTER_ALL,name)[1],
                             'n_observations':len(OUTER_ALL),
                             'fraction_changed':float(np.mean(abs(OUTER_ALL[name]-OUTER_ALL.baseline)>1e-8))}
                            for name in ['baseline','selected']])
MODEL_SUMMARY.to_csv(OUT/'nested_ablation_summary.csv',index=False)
print('\nNested ablation against Notebook 11:')
display(MODEL_SUMMARY)



## 5. Diagnostics, uncertainty and result archive

The output ZIP separates blind **coverage** (structure-only) from labeled OOF **validation** (MAE/ST-RAE). For challenge submission, first ensure external data use is permitted and report provenance; no notebook-generated blind predictions are written here.


In [ ]:
# Post-hoc fixed-candidate diagnostics are explicitly NOT independently selected validation estimates.
POSTHOC=get_summary(OUTER_ALL,ALL_CANDIDATES)
POSTHOC.to_csv(OUT/'outer_candidate_sensitivity_EXPLORATORY.csv',index=False)
PER_ISO=pd.concat([metric(OUTER_ALL,'baseline').assign(model='baseline'),
                   metric(OUTER_ALL,'selected').assign(model='selected')],ignore_index=True)
PER_ISO.to_csv(OUT/'nested_ablation_by_isoform.csv',index=False)
if not FAST_MODE:
    BOOT=bootstrap_scaffold(OUTER_ALL,reps=N_BOOTSTRAP,seed=SEED)
    BOOT.to_csv(OUT/'paired_scaffold_bootstrap.csv',index=False)
else:
    BOOT=pd.DataFrame()

ANALYSIS=[]
ref_coverage=COVERAGE_DETAILS[COVERAGE_DETAILS.dataset.eq('labeled_openadmet')].copy()
# Similarity-stratified evaluation, by isoform using chosen fingerprint / tier when available.
for iso,g in OUTER_ALL.groupby('isoform'):
    for category,lower,upper in [('<0.45',0,0.45),('0.45-0.55',.45,.55),('0.55-0.65',.55,.65),
                                  ('0.65-0.75',.65,.75),('>=0.75',.75,1.00001)]:
        coverage=ref_coverage[(ref_coverage.isoform.eq(iso)) & (ref_coverage.bits.eq(2048)) &
                              (ref_coverage.tier.eq('all'))][['canonical_smiles','nearest_similarity']]
        gg=g.merge(coverage,on='canonical_smiles',validate='one_to_one')
        gg=gg[gg.nearest_similarity.ge(lower)&gg.nearest_similarity.lt(upper)]
        if not len(gg):continue
        ANALYSIS.append({'isoform':iso,'similarity_bin':category,'n':len(gg),
          'baseline_mae':float(abs(gg.y_true-gg.baseline).mean()),
          'selected_mae':float(abs(gg.y_true-gg.selected).mean()),
          'mean_gain':float((abs(gg.y_true-gg.baseline)-abs(gg.y_true-gg.selected)).mean())})
pd.DataFrame(ANALYSIS).to_csv(OUT/'performance_by_chembl_similarity.csv',index=False)

META={'notebook':'15B_chembl_blind_coverage_reliability.ipynb','fast_mode':FAST_MODE,
      'real_chembl_data':bool(meta15['full_scientific_evaluation']),
      'blind_file':Path(BLIND_SOURCE).name,'blind_structure_sha256':sha_file(BLIND_SOURCE),
      'nb11_zip_sha256':sha_file(NB11_RESULTS_ZIP),'nb15_zip_sha256':sha_file(NB15_RESULTS_ZIP),
      'n_blind_structures':len(BLIND),'n_exact_blind_chembl_structures':BLIND_EXACT.canonical_smiles.nunique(),
      'n_outer_measurements':len(OUTER_ALL),'n_folds':len(FOLDS_TO_RUN),'settings':SIGNATURE,
      'cache_signature':RUN_HASH,'full_oof_evaluation':len(FOLDS_TO_RUN)==5 and not FAST_MODE and len(OUTER_ALL)==6525,
      'validation_policy':'Outer labels never used for tuning. Inner scaffolds disjoint from training calibration. Exact blind ChEMBL labels quarantined from references.',
      'notes':['Blind coverage is structural, NOT a blinded MA-ST-RAE score',
               'The source archive may contain ChEMBL assays published after the challenge cutoff',
               'Concordant tier excludes high cross-assay disagreements; single-assay observations are not independently validated',
               'Reported interval is conditional on trained models and frozen selection protocol',
               'Selection results use pIC50 from Notebook 11 OOF predictions as fixed base; no graph retraining']}
(OUT/'run_metadata.json').write_text(json.dumps(META,indent=2,default=str))
archive=BASE/'cyp_notebook_15B_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUT.glob('*')):
        if p.is_file():z.write(p,arcname='cyp_notebook_15B_results/'+p.name)
print('Wrote:',archive,'files:',len(list(OUT.glob('*'))),'complete:',META['full_oof_evaluation'])


### What to share for review

Upload **`cyp_notebook_15B_results.zip`**. The key files are:

- `coverage_comparison_train_vs_blind.csv`, `blind_chembl_coverage_by_isoform.csv`: distribution shift and per-CYP analog coverage.
- `assay_quality_filter_audit.csv`, `blind_exact_chembl_quarantine.csv`: reliability and exact-match provenance.
- `nested_ablation_summary.csv`, `nested_selected_params_by_fold.csv`, `paired_scaffold_bootstrap.csv`: incremental performance and stability.
- `outer_candidate_sensitivity_EXPLORATORY.csv`: exploratory threshold sensitivity (do **not** use its best row as the final estimate).
- `leakage_audit.csv`, `run_metadata.json`: split checks and completeness.

If the full nested approach rejects all corrections, that is scientifically useful: it means the strongest current baseline should be retained. Even when blind-set ChEMBL coverage is higher, no performance claim can be made without blind labels.
